# AV1 - Analise e Visualizacao de Dados (2026.2)
## Grupo 6 - EcoVolt | **Notebook 3 de 3 - V3: sensibilidade da decisao ao SOH**

**Objetivo.** Mostrar como a autonomia depende do **estado de saude (SOH)** da bateria de 2a vida, com base em um dataset **observado** de degradacao, e localizar o SOH minimo que atende ao horizonte.

**Pergunta (V3).** *Como muda a autonomia quando a premissa de SOH varia?*

**Estrutura do notebook.** 1) Coleta -> 2) Tratamento -> 3) Limpeza (5 verificacoes) -> 4) Indicador -> 5) Graficos -> 6) Por que estes graficos -> 7) Leitura e limitacoes.


## 1. Coleta de dados

**Fonte observada:** `dados/raw/ORIGINAL_nomeErrado_recarregados-emplacamentos.csv`, que - apesar do nome - e o **dataset SOH CALCE CS2** (grao = ciclo de uma celula; chave = `battery_id` + `cycle_number`).

> **Achado de qualidade:** o arquivo `..._CALCE_CS2_ML_SOH_Dataset.csv` **nao e CSV** - sua assinatura comeca com `PK` (planilha XLSX). Os arquivos chegaram com **nomes trocados**.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

def encontrar_raiz(inicio=None):
    """Sobe a arvore de diretorios ate achar a pasta do projeto (dados/ + analise/)."""
    p = Path(inicio or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "dados").exists() and (cand / "analise").exists():
            return cand
    raise FileNotFoundError("pasta do projeto nao encontrada")


BASE = encontrar_raiz()
RAW = BASE / "dados" / "raw"
PROC = BASE / "dados" / "processados"
VISTAS = BASE / "dashboard" / "vistas"
PROC.mkdir(parents=True, exist_ok=True)
VISTAS.mkdir(parents=True, exist_ok=True)
print("Raiz do projeto:", BASE)

suspeito = RAW / "ORIGINAL_nomeErrado_CALCE_CS2_ML_SOH_Dataset.csv"
print("Assinatura do '...CALCE_CS2_ML_SOH_Dataset.csv':", suspeito.read_bytes()[:4],
      "-> XLSX disfarcado de .csv")

calce_raw = pd.read_csv(RAW / "ORIGINAL_nomeErrado_recarregados-emplacamentos.csv")
cfg = json.loads((BASE / "analise" / "cenarios.json").read_text(encoding="utf-8"))
print("Dimensao do dataset CALCE:", calce_raw.shape)
calce_raw.head()


### 1.1 Dicionario de variaveis

| Variavel | Significado | Unidade |
|---|---|---|
| `battery_id` | identificador da celula | - |
| `cycle_number` | numero do ciclo de carga/descarga | contagem |
| `C_rate` | taxa de carga | C |
| `average_voltage_V` | tensao media do ciclo | V |
| `average_current_A` | corrente media do ciclo | A |
| `internal_resistance_ohm` | resistencia interna | ohm |
| `SOH` | estado de saude | % |


## 2. Tratamento

- Garantir a **chave unica** (`battery_id` + `cycle_number`).
- Criar `SOH_corrigido` limitando valores fisicamente impossiveis acima de 100%, mantendo o original e sinalizando com `soh_acima_100_flag`.


In [ ]:
calce = calce_raw.copy()
calce = calce.drop_duplicates(subset=["battery_id", "cycle_number"], keep="first")
calce["SOH_corrigido"] = calce["SOH"].clip(upper=100.0)
calce["soh_acima_100_flag"] = calce["SOH"] > 100
calce = calce.sort_values(["battery_id", "cycle_number"]).reset_index(drop=True)
print(f"linhas: bruto={len(calce_raw)} -> tratado={len(calce)}")
calce.head()


## 3. Limpeza - as cinco verificacoes

In [ ]:
def verificar(df, chave):
    """As cinco verificacoes de qualidade (bloco base)."""
    return {
        "1. chaves/duplicidade":
            f"{int(df.duplicated().sum())} linhas duplicadas; "
            f"{int(df.duplicated(subset=chave).sum())} duplicadas na chave {tuple(chave)}",
        "2. ausencia/codigos":
            f"{int(df.isna().sum().sum())} celulas nulas",
        "3. tipos/unidades":
            ", ".join(f"{c}={t}" for c, t in df.dtypes.astype(str).items()),
    }


def dominio_calce(df):
    return {
        "4. dominio/plausibilidade": (
            f"SOH {df['SOH'].min():.1f}..{df['SOH'].max():.1f} %; "
            f"V {df['average_voltage_V'].min():.2f}..{df['average_voltage_V'].max():.2f}; "
            f"R {df['internal_resistance_ohm'].min():.3f}..{df['internal_resistance_ohm'].max():.3f} ohm"),
        "5. cobertura/juncoes": (
            f"{df['battery_id'].nunique()} baterias ({', '.join(sorted(df['battery_id'].unique()))}); "
            f"C_rate {sorted(df['C_rate'].unique())}"),
    }


relatorio = {**verificar(calce_raw, ["battery_id", "cycle_number"]), **dominio_calce(calce_raw)}
relatorio["2. ausencia/codigos"] += f"; SOH > 100: {int((calce_raw['SOH'] > 100).sum())}"
pd.DataFrame(relatorio.items(), columns=["verificacao", "resultado"])


In [ ]:
decisoes = pd.DataFrame([
    {"problema": "Linhas duplicadas", "deteccao": "df.duplicated()",
     "afetados": int(calce_raw.duplicated().sum()),
     "regra": "manter a primeira", "consequencia": "sem perda"},
    {"problema": "Duplicidade na chave", "deteccao": "subset=[battery_id, cycle_number]",
     "afetados": int(calce_raw.duplicated(subset=["battery_id", "cycle_number"]).sum()),
     "regra": "manter a primeira", "consequencia": "chave unica confirmada"},
    {"problema": "SOH > 100% (impossivel)", "deteccao": "SOH > 100",
     "afetados": int((calce_raw["SOH"] > 100).sum()),
     "regra": "manter a linha, limitar a 100 em SOH_corrigido + flag",
     "consequencia": "ciclo preservado, valor sinalizado"},
    {"problema": "Celulas nulas", "deteccao": "isna()",
     "afetados": int(calce_raw.isna().sum().sum()),
     "regra": "descartar a linha", "consequencia": "nenhuma removida"},
])

calce_raw.to_csv(PROC / "calce_cs2_soh.csv", index=False)
calce.to_csv(PROC / "calce_cs2_soh_limpo.csv", index=False)
print(f"SOH corrigido: {calce['SOH_corrigido'].min():.1f}% a {calce['SOH_corrigido'].max():.1f}%")
decisoes


## 4. Indicador

O SOH observado justifica a premissa `fator_capacidade` e define a **faixa de sensibilidade** (0,60 a 1,00):

```
Eutilizavel = Enom * fator_capacidade * fracao_uso * eficiencia
autonomia   = Eutilizavel / Pcritica
```


In [ ]:
fatores = cfg["fatores_bateria"]
fracao_uso = fatores["fracao_uso"]["valor"]
eficiencia = fatores["eficiencia"]["valor"]
soh_adotado = fatores["fator_capacidade"]["valor"]
telecom = next(cen for cen in cfg["cenarios"] if cen["id"] == "telecom_erb")
pcritica_telecom = sum(x["potencia_kW"] * x["qtd"] for x in telecom["cargas"])

print(f"SOH observado (CALCE): {calce['SOH_corrigido'].min():.1f}% a {calce['SOH_corrigido'].max():.1f}%")
print(f"SOH adotado como premissa: {soh_adotado:.2f}")
print(f"Pcritica (telecom): {pcritica_telecom:.2f} kW")


## 5. Visualizacoes

**5.1 Evidencia de coleta/limpeza** - degradacao do SOH por ciclo (dado observado).

In [ ]:
import plotly.graph_objects as go

fig_soh = go.Figure()
for bid, g in calce.groupby("battery_id"):
    fig_soh.add_trace(go.Scatter(x=g["cycle_number"], y=g["SOH_corrigido"],
                                 mode="lines", name=bid))
fig_soh.update_layout(title="SOH por ciclo - CALCE CS2 (dado observado, apos limpeza)",
                      xaxis_title="Numero do ciclo", yaxis_title="SOH (%)",
                      template="plotly_white", height=460, legend_title="Bateria")
fig_soh.show()
fig_soh.write_html(VISTAS / "soh_por_ciclo.html", include_plotlyjs="cdn")


**5.2 Analise de decisao (V3)** - relacao entre `fator_capacidade` (SOH) e autonomia no cenario de telecom, com o horizonte de 4 h.

In [ ]:
faixa = np.round(np.linspace(0.6, 1.0, 21), 3)
fig = go.Figure()
cores = {"A": "#E1A100", "B": "#0B6E4F"}
for b in cfg["bancos"]:
    eu = b["Enominal_kWh"] * faixa * fracao_uso * eficiencia
    fig.add_trace(go.Scatter(x=faixa, y=eu / pcritica_telecom, mode="lines+markers",
                             name=f"Banco {b['id']} ({b['Enominal_kWh']:.0f} kWh)",
                             line=dict(color=cores[b["id"]], width=3)))
fig.add_hline(y=telecom["duracao_h"], line=dict(dash="dot", color="#333333"),
              annotation_text=f"Horizonte = {telecom['duracao_h']:.0f} h")
fig.add_vline(x=soh_adotado, line=dict(dash="dash", color="#8A8F98"),
              annotation_text=f"SOH adotado = {soh_adotado:.2f}")
fig.update_xaxes(title_text="Fator de capacidade remanescente (SOH) [adimensional]")
fig.update_yaxes(title_text="Autonomia (h)", rangemode="tozero")
fig.update_layout(title="V3. Sensibilidade da autonomia ao SOH (cenario telecom)",
                  template="plotly_white", height=520)
fig.show()
fig.write_html(VISTAS / "v3_sensibilidade.html", include_plotlyjs="cdn")


## 6. Por que estes graficos sao a melhor opcao?

**5.1 (SOH por ciclo) - tarefa:** mostrar **evolucao continua** de uma grandeza ao longo de muitos pontos (centenas/milhares de ciclos por bateria).

- **Grafico de linhas** e o padrao para series continuas: revela **tendencia e velocidade** de degradacao que numeros isolados esconderiam.
- **Uma linha por bateria** (poucas series, mesma escala %) compara sem poluir; um dispersao/hexbin perderia a noção de sequencia.

**5.2 (V3) - tarefa:** mostrar uma **relacao** entre uma variavel continua de entrada (SOH) e uma saida (autonomia), marcando onde a decisao **muda de lado** (cruza o horizonte).

- **Linhas + marcadores** com x continuo mostram a tendencia e o ponto de cruzamento; os marcadores dao os valores amostrados.
- `add_hline` (horizonte) e `add_vline` (SOH adotado) transformam **criterio e premissa em referencias visiveis** - o "ponto de virada" fica explicito.

**Alternativas descartadas**

| Alternativa | Por que nao |
|---|---|
| Barras para o SOH por ciclo | X e continuo com milhares de pontos; barras ficariam ilegiveis. |
| Dispersao pura na V3 | Perde a tendencia; o cruzamento com o horizonte fica menos claro. |
| Barras/heatmap na V3 | Nao ha categorias nem matriz; a relacao e funcional e continua. |


## 7. Leitura, decisao e limitacoes

- **Leitura:** o SOH observado (CALCE CS2) cai de ~100% a ~5% ao longo dos ciclos, sustentando o uso de uma faixa realista (0,60-1,00) na sensibilidade. No cenario de telecom, o **Banco B** cruza o horizonte de 4 h por volta de **SOH ~0,60**, enquanto o **Banco A** so o alcancaria com SOH muito proximo de 1,0 - inviavel na pratica.
- **Decisao apoiada:** a escolha do Banco B e **robusta** a incerteza de SOH; a do Banco A e fragil.
- **Limite:** o dataset CALCE e de **celulas de laboratorio** (CS2, 5,596 Ah) em condicoes controladas; nao cobre envelhecimento calendario, temperatura nem a montagem do banco. O `fator_capacidade = 0,80` e premissa de 2a vida, **nao** substitui ensaio de capacidade do banco montado.
